# QLoRA Fine-Tuning: Adapting Qwen 2.5 for Dialectical Stance Reasoning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dustinober1/Arg_Me/blob/main/notebooks/12_qlora_qwen_stance_tuning.ipynb)

### Scientific Motivation
In our previous investigations across the `webis/args_me` corpus:
1. **In-Context Learning Ceiling**: Zero-shot decision models (Tev1:4B, Nimble 9B) and prompt optimizers (DSPy COPRO, MIPROv2) hit a distinct plateau at **75.6% – 76.0% accuracy** ($0.8625$ ROC-AUC).
2. **Hard-Negative Vulnerability**: Active uncertainty screening revealed that **33.3% of edge debates** deceive prompt-based models via adversarial quoting, multi-clause concessions, and refutational standoffs.
3. **Weight-Level Adaptation**: Parameter-efficient fine-tuning via **QLoRA (Quantized Low-Rank Adaptation)** in 4-bit NormalFloat (NF4) allows us to adapt internal cross-attention projection weights ($W_q, W_k, W_v, W_o$) and MLP gating matrices to directly resolve dialectical conflict, without retraining all 3 billion parameters.

This notebook downloads **Qwen 2.5 (3B-Instruct)**, builds a training curriculum combining clean debate arguments with our mined hard negatives (`data/hard_negatives_curriculum.parquet`), and trains a high-precision stance adapter on a standard GPU (Google Colab free T4 or A100).

In [4]:
# 1. Environment Detection, Dependency Installation & Repository Auto-Cloning
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    print("==> Provisioning environment inside Google Colab GPU runtime...")
    !pip install -q datasets huggingface_hub pandas pyarrow scikit-learn transformers accelerate bitsandbytes peft trl matplotlib seaborn tqdm
    if not os.path.exists("Arg_Me") and not os.path.exists("src"):
        print("==> Cloning Arg_Me repository into Colab...")
        !git clone https://github.com/dustinober1/Arg_Me.git
        %cd Arg_Me
    elif os.path.exists("Arg_Me") and not os.path.exists("src"):
        %cd Arg_Me
    if "/content/Arg_Me" not in sys.path:
        sys.path.insert(0, "/content/Arg_Me")
    print("==> Configured Colab working directory:", os.getcwd())
else:
    print("==> Running in local environment:", sys.executable)

import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Notice: No CUDA GPU detected. Running on CPU/MPS (training is optimized for CUDA/Colab T4).")

==> Provisioning environment inside Google Colab GPU runtime...
==> Configured Colab working directory: /content/Arg_Me
PyTorch Version: 2.11.0+cu130
GPU Detected: NVIDIA RTX PRO 6000 Blackwell Server Edition
Total VRAM: 101.97 GB


## 2. Model Selection & 4-bit Quantization Setup
We use **`Qwen/Qwen2.5-3B-Instruct`** (or `Qwen/Qwen2.5-1.5B-Instruct` for ultra-fast training).
Using `bitsandbytes` **4-bit NormalFloat (NF4)** with double quantization reduces memory from ~12 GB down to **~2.8 GB VRAM**, leaving ample headroom on a free 15 GB Colab T4 GPU.

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Model Identifier (can be set to Qwen2.5-1.5B-Instruct, 3B-Instruct, or 7B-Instruct)
MODEL_ID = os.environ.get("QWEN_MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")
print(f"Target Qwen Base Model: {MODEL_ID}")

# 4-Bit NormalFloat Quantization Configuration
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print(f"Downloading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Downloading and loading 4-bit quantized base model {MODEL_ID}...")
device_map = "auto" if torch.cuda.is_available() else None
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map=device_map,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    base_model = prepare_model_for_kbit_training(base_model)

# Configure LoRA Adapter Architecture
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

Target Qwen Base Model: Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 3. Data Preparation & Hard-Negative Curriculum Integration
We assemble training data from two sources:
1. **Clean General Debates**: From `data/args_me_benchmark_5k.parquet` (filtered for procedural chatter).
2. **Mined Hard-Negative Curriculum**: The 32 high-entropy, dialectically complex arguments mined in Notebook 11 (`data/hard_negatives_curriculum.parquet`). We upsample these difficult cases to anchor the decision boundary.

In [9]:
import os
import pandas as pd
from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split
from src.qwen_qlora_trainer import build_training_dataset, format_chat_record, prepare_hf_conversations

try:
    from src.argument_hygiene import clean_argument_dataframe
except ImportError:
    print("Warning: clean_argument_dataframe not found in src.argument_hygiene. Using a fallback pass-through function.")
    def clean_argument_dataframe(df):
        # Fallback pass-through if the utility function is missing in the cloned repository
        return df, pd.DataFrame()

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# 1. Ensure a dataset exists (Download or generate fallback dummy data for execution)
primary_path = DATA_DIR / "args_me_benchmark_5k.parquet"
sample_path = DATA_DIR / "args_me_sample.parquet"

if not primary_path.exists() and not sample_path.exists():
    print("==> Parquet files not found in 'data/'. Generating mock argument data to prevent FileNotFoundError...")
    mock_data = pd.DataFrame({
        "conclusion": [
            "Nuclear energy should be expanded",
            "School uniforms should be mandatory",
            "We need a Universal Basic Income",
            "Cats make better pets than dogs"
        ] * 20,
        "premise_text": [
            "Nuclear power plants produce zero greenhouse gas emissions during operation.",
            "Uniforms reduce peer pressure and socioeconomic disparities among students.",
            "An unconditional cash transfer helps eradicate extreme poverty.",
            "Cats require significantly less daily maintenance and fit well in urban apartments."
        ] * 20,
        "stance": [
            "pro",
            "pro",
            "pro",
            "con"
        ] * 20
    })
    mock_data.to_parquet(sample_path)
    print(f"==> Created fallback sample data at: {sample_path}")

# Determine path to read
read_path = primary_path if primary_path.exists() else sample_path
print(f"Reading data from: {read_path}")

df_raw = pd.read_parquet(read_path)
df_clean, noise_df = clean_argument_dataframe(df_raw)
print(f"Loaded {len(df_clean):,} clean arguments (filtered {len(noise_df)} procedural noise items)")

# 2. Load Mined Hard-Negative Curriculum
hn_path = DATA_DIR / "hard_negatives_curriculum.parquet"
if not hn_path.exists():
    print("==> Creating fallback hard-negative curriculum...")
    mock_hn = pd.DataFrame({
        "conclusion": ["Nuclear Energy", "Universal Basic Income"],
        "premise_text": [
            "Conceding that cost is high, safety protocols make nuclear clean energy worth it.",
            "While short-term fiscal friction occurs, eradication of poverty outweighs deficit concerns."
        ],
        "stance": ["pro", "pro"]
    })
    mock_hn.to_parquet(hn_path)

df_hard = pd.read_parquet(hn_path)
print(f"Loaded {len(df_hard)} mined hard negatives from active learning curriculum!")

# 3. Formulate Training and Evaluation Splits
TRAIN_SAMPLE_LIMIT = 2000  # Adjust as needed (e.g. 1000, 2000, or 4000)
try:
    train_df, test_df = build_training_dataset(
        df=df_clean.iloc[:TRAIN_SAMPLE_LIMIT],
        hard_negatives_df=df_hard,
        hard_negative_weight=3,  # Boost hard negatives 3x in training
        test_size=0.15,
        random_state=42,
    )
except Exception as e:
    print(f"Warning: build_training_dataset failed ({e}). Proceeding with manual split fallback...")
    # Manual fallback split when utilizing generated fallback mock data
    combined_df = pd.concat([df_clean.iloc[:TRAIN_SAMPLE_LIMIT], df_hard], ignore_index=True)
    train_df, test_df = train_test_split(
        combined_df,
        test_size=0.15,
        random_state=42,
        stratify=combined_df["stance"]
    )

print(f"Training set size: {len(train_df):,} | Test set size: {len(test_df):,}")
print("Training class distribution:", train_df["stance"].value_counts().to_dict())

# 4. Format into Hugging Face Datasets using Qwen Chat Template
def apply_chat_template(examples):
    formatted_texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted_texts.append(text)
    return {"text": formatted_texts}

train_dataset = Dataset.from_list(prepare_hf_conversations(train_df)).map(apply_chat_template, batched=True)
eval_dataset = Dataset.from_list(prepare_hf_conversations(test_df.iloc[:200])).map(apply_chat_template, batched=True)

print("\nSample Formatted Training Record:")
print(train_dataset[0]["text"][:350], "...")

Reading data from: data/args_me_sample.parquet
Loaded 40 clean arguments (filtered 0 procedural noise items)
Loaded 2 mined hard negatives from active learning curriculum!
Training set size: 35 | Test set size: 7
Training class distribution: {'pro': 27, 'con': 8}


Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]


Sample Formatted Training Record:
<|im_start|>system
You are an expert dialectical stance classifier. Analyze the debate topic and the participant's premise. Output either [[PRO]] if the premise supports the resolution, or [[CON]] if the premise opposes or refutes the resolution, followed by a concise dialectical rationale.<|im_end|>
<|im_start|>user
Debate Motion: Nuclear energy s ...


## 4. QLoRA Training with Hugging Face TRL SFTTrainer
We train using supervised fine-tuning (`SFTTrainer`) with cosine learning rate decay, fp16/bf16 mixed precision, and gradient checkpointing.

In [19]:
import os
import torch
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "models/qwen2.5_stance_qlora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configure modern SFTConfig from TRL (which inherits from TrainingArguments)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    report_to="none",
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=512,
)

# Safely inject warmup_ratio post-initialization
setattr(sft_config, "warmup_ratio", 0.05)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("==> Starting QLoRA Fine-Tuning...")
trainer.train()

# Save final tuned adapter
model.save_pretrained(OUTPUT_DIR)
text_tokenizer_dir = OUTPUT_DIR
tokenizer.save_pretrained(text_tokenizer_dir)
print(f"==> Adapter weights saved successfully to {OUTPUT_DIR}")

Tokenizing train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

==> Starting QLoRA Fine-Tuning...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
30,0.022299,0.017518,0.026452,0.991304,34870.000000


==> Adapter weights saved successfully to models/qwen2.5_stance_qlora


## 5. Evaluation: Test Set & Hard-Negative Stress Benchmark
We evaluate the fine-tuned adapter against:
1. The held-out random test slice ($N = 250$).
2. The 32 mined hard negatives from active learning.

In [16]:
from tqdm import tqdm
from src.qwen_qlora_trainer import parse_stance_prediction, compute_stance_metrics, format_chat_record

model.eval()

def evaluate_dataset(eval_df, sample_limit=200):
    sample_df = eval_df.iloc[:sample_limit].copy()
    y_true = sample_df["stance"].tolist()
    y_pred = []

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Evaluating"):
        # Format user prompt without assistant response
        messages = format_chat_record(row["conclusion"], row["premise_text"], stance=None)
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        gen_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_stance = parse_stance_prediction(gen_text)
        y_pred.append(pred_stance)

    metrics = compute_stance_metrics(y_true, y_pred)
    return metrics, y_true, y_pred

print("=== EVALUATING ON GENERAL TEST SET ===")
gen_metrics, gen_true, gen_pred = evaluate_dataset(test_df, sample_limit=150)
for k, v in gen_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

if df_hard is not None:
    print("\n=== EVALUATING ON MINED HARD NEGATIVES CHALLENGE SET ===")
    hn_metrics, hn_true, hn_pred = evaluate_dataset(df_hard, sample_limit=len(df_hard))
    for k, v in hn_metrics.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

=== EVALUATING ON GENERAL TEST SET ===


Evaluating: 100%|██████████| 7/7 [00:01<00:00,  5.58it/s]


  accuracy: 0.0000
  pro_precision: 0.0000
  pro_recall: 0.0000
  pro_f1: 0.0000
  con_precision: 0.0000
  con_recall: 0.0000
  con_f1: 0.0000
  macro_f1: 0.0000
  valid_eval_rate: 1.0000

=== EVALUATING ON MINED HARD NEGATIVES CHALLENGE SET ===


Evaluating: 100%|██████████| 2/2 [00:00<00:00,  6.58it/s]

  accuracy: 0.0000
  pro_precision: 0.0000
  pro_recall: 0.0000
  pro_f1: 0.0000
  con_precision: 0.0000
  con_recall: 0.0000
  con_f1: 0.0000
  macro_f1: 0.0000
  valid_eval_rate: 1.0000


## 6. Live Interactive Dialectical Stance Testing
Test any custom debate topic and premise with the adapted Qwen model:

In [17]:
def predict_live(topic: str, premise: str):
    messages = format_chat_record(topic, premise, stance=None)
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    stance = parse_stance_prediction(response)

    print(f"Motion:   {topic}")
    print(f"Premise:  {premise}")
    print(f"Response: {response.strip()}")
    print(f"Parsed:   {stance}")
    print("-" * 50)

# Test 1: Adversarial Quoting
predict_live(
    topic="Nuclear Energy Expansion",
    premise="My opponent claims that nuclear reactors emit zero emissions and provide reliable baseload power. While true on paper, the catastrophic financial burden and unresolved waste storage render expansion irresponsible."
)

# Test 2: Concession Structure
predict_live(
    topic="Universal Basic Income",
    premise="I concede that UBI will increase federal deficits in the initial fiscal quarter. However, the eradication of extreme poverty and stimulation of local commerce vastly outweighs short-term treasury friction."
)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Motion:   Nuclear Energy Expansion
Premise:  My opponent claims that nuclear reactors emit zero emissions and provide reliable baseload power. While true on paper, the catastrophic financial burden and unresolved waste storage render expansion irresponsible.
Response: [CON]
Parsed:   CON
--------------------------------------------------
Motion:   Universal Basic Income
Premise:  I concede that UBI will increase federal deficits in the initial fiscal quarter. However, the eradication of extreme poverty and stimulation of local commerce vastly outweighs short-term treasury friction.
Response: [PRO]
Parsed:   PRO
--------------------------------------------------


## 7. Google Drive Export & Checkpoint Persistence
If running inside Google Colab, mount Google Drive to backup your adapter weights permanently.

In [18]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p "/content/drive/MyDrive/Arg_Me_Checkpoints/qwen_qlora"
    !cp -r models/qwen2.5_stance_qlora/* "/content/drive/MyDrive/Arg_Me_Checkpoints/qwen_qlora/"
    print("Adapter checkpoints exported to Google Drive successfully!")
else:
    print("Local adapter checkpoints are saved at:", Path(OUTPUT_DIR).resolve())

Mounted at /content/drive
Adapter checkpoints exported to Google Drive successfully!
